# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayesha-asif1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row in fact_content_daily_performance represents one page's
(content_hash_id) search performance for one client (client_hash_id)
on one specific day (report_date).

My time window is March 2026 — report_date between 2026-03-01 and 2026-03-31.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded
**Label**
- CTR = gsc_clicks / gsc_impressions
  (computed from the two raw columns, not a column in the table itself)

**Features**
- gsc_impressions — how many times the page was shown (known before any click happens)
- gsc_avg_position — the page's average search ranking position that day

**Context**
- client_hash_id — identifies which client (not a model input, used to group/filter)
- content_hash_id — identifies which page (not a model input, used to group/filter)
- report_date — identifies which day (used for filtering the time window)
- gsc_data_available — used to filter to reliable rows, not a model input

**Excluded**
- gsc_clicks — excluded as a feature because it is used to compute the label itself
  (CTR = clicks/impressions). Including it as an input would leak the answer into
  the model and produce artificially perfect scores.
- gsc_sum_position — excluded because it is redundant with gsc_avg_position and
  harder to interpret on its own.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


3. **Verify it with queries (grain, counts, missing values, windows)**
Every claim above gets a query cell here. A contract claim without a query next to it is a guess.

In [ ]:
from google.colab import userdata
hf_token=userdata.get('HF_TOKEN')

In [ ]:
!pip install huggingface_hub pandas pyarrow -q


In [ ]:
from huggingface_hub import login
login(token=hf_token)

In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=hf_token
)

# Sirf fact_content_daily_performance wali files dikhao
for f in files:
    if "daily_performance" in f:
        print(f)

fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/month=2026-04/data_0.parquet
fact_content_daily_performance/month=202

In [ ]:
import pandas as pd

df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("Shape:", df_march.shape)
df_march.head()

Shape: (9841378, 31)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-03


In [ ]:
#Grain_Check
duplicate_count=df_march.duplicated(
    subset=['client_hash_id','content_hash_id','report_date']
).sum()

print(f"Number of Duplicate(client,content,date) rows:{duplicate_count}")

Number of Duplicate(client,content,date) rows:0


In [ ]:
#Shape Check(row count+date span)
print(f" Total rows in March 2026:{len(df_march)}")
print(f"Earliest date: {df_march['report_date'].min()}")
print(f"Latest date: {df_march['report_date'].max()}")
## Shape check: confirms full March 2026 coverage with no missing/extra dates

 Total rows in March 2026:9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31


In [ ]:
#Availability
available_rows=df_march[df_march['gsc_data_available']==True]
print(f"Total Rows: {len(df_march)}")
print(f"Rows with gsc_data_available = True: {len(available_rows)}")
print(f"Percentage available: {len(available_rows) / len(df_march) * 100:.2f}%")

Total Rows: 9841378
Rows with gsc_data_available = True: 3611061
Percentage available: 36.69%


In [ ]:
df_available = df_march[df_march['gsc_data_available'] == True].copy()
print(f"Working dataset shape: {df_available.shape}")

Working dataset shape: (3611061, 31)


In [ ]:
zero_impressions=(df_available['gsc_impressions']==0).sum()
print(f"Number of rows with gsc_impressions=0: {zero_impressions}")
print(f"rows with gsc_impressions=0: {zero_impressions}")

Number of rows with gsc_impressions=0: 0
rows with gsc_impressions=0: 0


In [ ]:
#Label

df_available['ctr']=df_available['gsc_clicks'] /df_available['gsc_impressions']
print(df_available[['gsc_clicks','gsc_impressions','ctr']].head())

   gsc_clicks  gsc_impressions    ctr
0           0               20  0.000
1           0                1  0.000
2           1              125  0.008
3           0                7  0.000
4           0               11  0.000


**Feature 1**

day_of_week is knowable at the decision moment because the date is always known in advance — it doesn't depend on any future outcome."


In [ ]:
#day_of_week is knowable at the decision moment because the date is always known in advance — it doesn't depend on any future outcome."
df_available['day_of_week'] = pd.to_datetime(df_available['report_date']).dt.dayofweek

**Feature 2**

gsc_avg_position is knowable at the decision moment because it reflects the page's search ranking position at that time — it's an input signal, not derived from clicks.

In [ ]:
df_available[['gsc_avg_position']].head()

,gsc_avg_position
0,3.350000
1,0.000000
2,4.928000
3,4.000000
4,2.272727


**Feature 3**

log_impressions is knowable at the decision moment because it's just a mathematical transform of gsc_impressions, which is already available.

In [ ]:
import numpy as np
df_available['log_impressions'] = np.log1p(df_available['gsc_impressions'])

**Feature 4**

position_bucket is knowable at the decision moment because it's a direct categorization of gsc_avg_position, which is already an input signal

In [ ]:
def position_bucket(pos):
    if pos <= 3:
        return 'top_3'
    elif pos <= 10:
        return 'top_10'
    elif pos <= 20:
        return 'top_20'
    else:
        return 'beyond_20'

df_available['position_bucket'] = df_available['gsc_avg_position'].apply(position_bucket)

**Feature 5**

gsc_impressions is knowable at the decision moment because it represents how many times the page was already shown in search results — this is recorded independently of whether anyone clicked. It doesn't depend on the label (clicks), so it's safe to use as an input.


In [ ]:
df_available[['gsc_impressions']].head()

,gsc_impressions
0,20
1,1
2,125
3,7
4,11


In [ ]:
feature_frame = df_available[[
    'client_hash_id', 'content_hash_id', 'report_date',   # context (identify karne ke liye)
    'ctr',                                                  # label
    'gsc_impressions', 'day_of_week', 'gsc_avg_position',   # features
    'log_impressions', 'position_bucket'                    # features
]].copy()

print(feature_frame.shape)
feature_frame.head()

(3611061, 9)


,client_hash_id,content_hash_id,report_date,ctr,gsc_impressions,day_of_week,gsc_avg_position,log_impressions,position_bucket
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,0.000,20,6,3.350000,3.044522,top_10
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,0.000,1,6,0.000000,0.693147,top_3
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,0.008,125,6,4.928000,4.836282,top_10
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,0.000,7,6,4.000000,2.079442,top_10
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,0.000,11,6,2.272727,2.484907,top_3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.